# Prepare Data Inputs 

Input parameters as requested:

OHLC Data (Open, High, Low, Close)
- No volume used
- Add Indicators:
  - EMA(5), EMA(10)
  - RSI
  - MACD Histogram
  - ADX
  - ATR
  - Bollinger Band Width
  - Candle Body/Wick Ratio
  - Previous 3 Candle Trend (+2 = 2 UP, -2 = 2 DOWN)

  Anticipated calculation:

  EMA(5) & EMA(10):
  Common Smoothning factor: 2
  Multiplier would be 0.3333 for EMA(5) and 0.1818 EMA(10) Formula [Common factor/(Number of periods(in days) + 1)]
  EMA = (closing price) * Multiplier + Previous EMA * (1 - Multiplier) 

In [34]:
# Importing raw data 

import pandas as pd

In [35]:
col_names = ['timestamp', 'open', 'high', 'low', 'close', 'volume']

In [36]:
raw_df = pd.read_excel("../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/RECENT_DATA_FILE_DUMP_EURUSD_M1.xlsx", header = None)

In [43]:
df = raw_df[0].str.split(";", expand=True)
df.columns = col_names
df['timestamp'] = pd.to_datetime(df['timestamp'], format="%Y%m%d %H%M%S")

# Convert price columns to float
df[['open', 'high', 'low', 'close']] = df[['open', 'high', 'low', 'close']].astype(float)

clean_df = df.drop(columns=['volume'])

print(clean_df.head())
print(df.index.is_unique)
print(df.index.duplicated())

            timestamp     open     high      low    close
0 2023-01-01 17:04:00  1.06970  1.06974  1.06970  1.06970
1 2023-01-01 17:05:00  1.06973  1.06978  1.06970  1.06971
2 2023-01-01 17:06:00  1.06966  1.06966  1.06966  1.06966
3 2023-01-01 17:08:00  1.06970  1.06974  1.06970  1.06974
4 2023-01-01 17:10:00  1.06975  1.06980  1.06972  1.06972
True
[False False False ... False False False]


In [44]:
# Set timestamp as index (critical for resampling)
clean_df.set_index('timestamp', inplace=True)
clean_df.index = pd.to_datetime(clean_df.index)
print(clean_df.index.duplicated().sum())
# To see duplicate values:
print(clean_df.index[clean_df.index.duplicated()])
clean_df = clean_df[~clean_df.index.duplicated(keep='first')]

60
DatetimeIndex(['2023-10-29 19:00:00', '2023-10-29 19:01:00',
               '2023-10-29 19:02:00', '2023-10-29 19:03:00',
               '2023-10-29 19:04:00', '2023-10-29 19:05:00',
               '2023-10-29 19:06:00', '2023-10-29 19:07:00',
               '2023-10-29 19:08:00', '2023-10-29 19:09:00',
               '2023-10-29 19:10:00', '2023-10-29 19:11:00',
               '2023-10-29 19:12:00', '2023-10-29 19:13:00',
               '2023-10-29 19:14:00', '2023-10-29 19:15:00',
               '2023-10-29 19:16:00', '2023-10-29 19:17:00',
               '2023-10-29 19:18:00', '2023-10-29 19:19:00',
               '2023-10-29 19:20:00', '2023-10-29 19:21:00',
               '2023-10-29 19:22:00', '2023-10-29 19:23:00',
               '2023-10-29 19:24:00', '2023-10-29 19:25:00',
               '2023-10-29 19:26:00', '2023-10-29 19:27:00',
               '2023-10-29 19:28:00', '2023-10-29 19:29:00',
               '2023-10-29 19:30:00', '2023-10-29 19:31:00',
               '2023-

In [39]:
#Combine Text File generator
import os
text_file_directory = "../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/Text Files/"

def get_dir_files(text_file_directory: str) -> list: 
    """
    Syntax: os.listdir(path)   
    Parameters: path (optional) :  path of the directory  
    Return Type: This method returns the list of all files and directories in the specified path. The return type of this method is list. 
    """
    all_files = os.listdir(text_file_directory)
    return all_files

def combine_text_files(all_files: list):
    print(f"{all_files}")
    with open(f'{text_file_directory}/combined_txt_file.txt', 'a') as file:
        for a_file in all_files:
            print(f"{a_file}")
            with open(f'{text_file_directory}/{a_file}', 'r') as temp_file:
                file.write(temp_file.read() + '\n')
    return f'{text_file_directory}/combined_txt_file.txt'

all_files = get_dir_files(text_file_directory)
combine_text_file_path = combine_text_files(all_files)

print(f"New file created: {combine_text_file_path}")

['combined_txt_file.txt', 'DAT_ASCII_EURUSD_M1_2023.txt', 'DAT_ASCII_EURUSD_M1_2024.txt', 'DAT_ASCII_EURUSD_M1_202501.txt', 'DAT_ASCII_EURUSD_M1_202502.txt', 'DAT_ASCII_EURUSD_M1_202503.txt', 'DAT_ASCII_EURUSD_M1_202504.txt']
combined_txt_file.txt
DAT_ASCII_EURUSD_M1_2023.txt
DAT_ASCII_EURUSD_M1_2024.txt
DAT_ASCII_EURUSD_M1_202501.txt
DAT_ASCII_EURUSD_M1_202502.txt
DAT_ASCII_EURUSD_M1_202503.txt
DAT_ASCII_EURUSD_M1_202504.txt
New file created: ../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/Text Files//combined_txt_file.txt


In [ ]:
#Handling gap data to either forward fill missing data or completely remove large gaps in data
import re

def parse_gap_report(file_path):
    gaps = []
    with open(file_path, 'r') as f:
        for line in f:
            match = re.match(r"Gap of (\d+)s found between (\d{14}) and (\d{14})\.", line)
            if match:
                duration = int(match.group(1))
                start = pd.to_datetime(match.group(2), format="%Y%m%d%H%M%S")
                end = pd.to_datetime(match.group(3), format="%Y%m%d%H%M%S")
                gaps.append({"start": start, "end": end, "duration_s": duration})
    return gaps

def handling_gaps(clean_df, combine_text_file_path):
    temp = []
    missing_values = pd.DataFrame(temp)
    if combine_text_file_path:
        gaps = parse_gap_report(combine_text_file_path)
        # Forward fill small gaps (<= 300s) at 1-minute level
        clean_df = clean_df.asfreq('1T', method='ffill')
        # Flag large gaps (> 300s)
        for gap in gaps:
            if gap['duration_s'] > 300:
                # Mark period as unreliable (e.g., set to NaN or flag)
                clean_df.loc[gap['start']:gap['end']] = None
                missing_values = pd.DataFrame([gap['start'], gap['end']])
    else:
        # Forward fill all gaps if no gap report
        clean_df = clean_df.asfreq('1T', method='ffill')    
    return clean_df, missing_values


In [46]:

clean_df, missing_values_df = handling_gaps(clean_df, combine_text_file_path)
df_1min = clean_df.dropna()
print("\n1-Minute Data:")
print(df_1min.head())

# RESAMPLING 3-minute OHLC
df_3min = clean_df.resample('3T').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last'
}).dropna()

# RESAMPLING 5-minute OHLC
df_5min = clean_df.resample('5T').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last'
}).dropna()

# Preview
print("3-Minute Data:")
print(df_3min.head())

print("\n5-Minute Data:")
print(df_5min.head())



1-Minute Data:
                        open     high      low    close
timestamp                                              
2023-01-01 17:04:00  1.06970  1.06974  1.06970  1.06970
2023-01-01 17:05:00  1.06973  1.06978  1.06970  1.06971
2023-01-01 17:06:00  1.06966  1.06966  1.06966  1.06966
2023-01-01 17:07:00  1.06966  1.06966  1.06966  1.06966
2023-01-01 17:08:00  1.06970  1.06974  1.06970  1.06974
3-Minute Data:
                        open     high      low    close
timestamp                                              
2023-01-01 17:03:00  1.06970  1.06978  1.06970  1.06971
2023-01-01 17:06:00  1.06966  1.06974  1.06966  1.06974
2023-01-01 17:09:00  1.06970  1.06980  1.06970  1.06972
2023-01-01 17:12:00  1.06975  1.07066  1.06899  1.06899
2023-01-01 17:15:00  1.06788  1.06788  1.06788  1.06788

5-Minute Data:
                        open     high      low    close
timestamp                                              
2023-01-01 17:00:00  1.06970  1.06974  1.06970  1.06970
2

In [49]:
print(missing_values_df.head(10))

                    0
0 2025-03-30 18:59:59
1 2025-03-30 20:00:00
